# NB4 — Resultados y Análisis

**Plataforma:** Vast.ai / local — no requiere GPU.

## Propósito
Cargar todos los resultados de NB2 y NB3, producir las tablas finales
del paper, calcular los hallazgos clave y exportar un CSV de resultados.

## Prerequisito
NB2 y NB3 deben haberse ejecutado completamente.
`nb2_results.json` y `nb3_results.json` deben existir.

## 1. Cargar resultados

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

RES_DIR = Path('/workspace/negative_supervision/results')

assert (RES_DIR / 'nb2_results.json').exists(), 'Ejecutar NB2 primero.'
assert (RES_DIR / 'nb3_results.json').exists(), 'Ejecutar NB3 primero.'

with open(RES_DIR / 'nb2_results.json') as f:
    NB2 = json.load(f)
with open(RES_DIR / 'nb3_results.json') as f:
    NB3 = json.load(f)

print('Resultados cargados.')
print(f'NB2: {list(NB2.keys())}')
print(f'NB3: {list(NB3.keys())}')

## 2. Consolidar resultados en DataFrame

In [ ]:
ROWS = []

# Resultados de NB2: BERT baseline y AAN random
NB2_MAP = {
    'baseline': ('BERT-base',  'Baseline'),
    'aan':      ('BERT-base',  'AAN (random)')
}
for ds in ['mr', 'semeval']:
    for key, (encoder, method) in NB2_MAP.items():
        if ds in NB2 and key in NB2[ds]:
            r = NB2[ds][key]
            ROWS.append({'Encoder': encoder, 'Método': method,
                         'Dataset': ds.upper(), 'Media': r['mean'],
                         'Std': r['std'], 'Mejor LR': r.get('best_lr', '-')})

# Resultados de NB3: BERT hard + RoBERTa
NB3_MAP = {
    'bert_aan_hard':    ('BERT-base',    'AAN (hard)'),
    'roberta_baseline': ('RoBERTa-base', 'Baseline'),
    'roberta_aan':      ('RoBERTa-base', 'AAN (random)'),
    'roberta_aan_hard': ('RoBERTa-base', 'AAN (hard)'),
}
for ds in ['mr', 'semeval']:
    for key, (encoder, method) in NB3_MAP.items():
        if ds in NB3 and key in NB3[ds]:
            r = NB3[ds][key]
            ROWS.append({'Encoder': encoder, 'Método': method,
                         'Dataset': ds.upper(), 'Media': r['mean'],
                         'Std': r['std'], 'Mejor LR': r.get('best_lr', '-')})

df = pd.DataFrame(ROWS)
print(f'Total de resultados: {len(df)}')
print(df.to_string(index=False))

## 3. Tabla principal de resultados

Formato listo para el paper. Filas = modelos, columnas = datasets.
El mejor resultado por dataset se marca con (*).

In [ ]:
ENCODER_ORDER = ['BERT-base', 'RoBERTa-base']
METHOD_ORDER  = ['Baseline', 'AAN (random)', 'AAN (hard)']
DATASET_ORDER = ['MR', 'SEMEVAL']
METRIC_NAMES  = {'MR': 'Accuracy', 'SEMEVAL': 'Exact Match'}

# Mejor resultado por dataset
best_per_ds = {ds: df[df['Dataset']==ds]['Media'].max() for ds in DATASET_ORDER}

print('=' * 80)
print('TABLA DE RESULTADOS PRINCIPALES')
print('=' * 80)
header = f'{"Encoder":<15} {"Método":<15}'
for ds in DATASET_ORDER:
    header += f'  {ds} ({METRIC_NAMES[ds]})'
print(header)
print('-' * 80)

prev_enc = None
for encoder in ENCODER_ORDER:
    if prev_enc is not None:
        print('-' * 80)
    prev_enc = encoder
    for method in METHOD_ORDER:
        sub = df[(df['Encoder']==encoder) & (df['Método']==method)]
        if len(sub) == 0:
            continue
        row = f'{encoder:<15} {method:<15}'
        for ds in DATASET_ORDER:
            entry = sub[sub['Dataset']==ds]
            if len(entry) > 0:
                mean = entry.iloc[0]['Media']
                std  = entry.iloc[0]['Std']
                mark = '(*)' if abs(mean - best_per_ds[ds]) < 1e-6 else '   '
                row += f'  {mean:.4f} ± {std:.4f} {mark}'
            else:
                row += f'  {"(faltante)":>22}'
        print(row)

print('=' * 80)
print('(*) = mejor resultado para ese dataset')

## 4. Análisis de hallazgos clave

Cuantificamos automáticamente los efectos de cada componente.

In [ ]:
def get(encoder, method, dataset):
    """Obtiene media para una configuración específica."""
    sub = df[(df['Encoder']==encoder) & (df['Método']==method) & (df['Dataset']==dataset)]
    return sub.iloc[0]['Media'] if len(sub) > 0 else None

print('HALLAZGOS CLAVE')
print('=' * 70)

for ds in DATASET_ORDER:
    metric = METRIC_NAMES[ds]
    print(f'\nDataset: {ds} | Métrica: {metric}')
    print('-' * 50)

    b_base  = get('BERT-base',    'Baseline',     ds)
    b_rand  = get('BERT-base',    'AAN (random)', ds)
    b_hard  = get('BERT-base',    'AAN (hard)',   ds)
    r_base  = get('RoBERTa-base', 'Baseline',     ds)
    r_rand  = get('RoBERTa-base', 'AAN (random)', ds)
    r_hard  = get('RoBERTa-base', 'AAN (hard)',   ds)

    # 1. Efecto de AAN random sobre BERT baseline
    if b_base and b_rand:
        d = b_rand - b_base
        pct = d / b_base * 100
        print(f'  1. BERT: AAN random vs Baseline:      {d:+.4f} ({pct:+.1f}%) '
              f'→ {"MEJORA" if d>0 else "DEGRADACIÓN"}')

    # 2. Efecto del hard mining sobre AAN random (BERT)
    if b_rand and b_hard:
        d = b_hard - b_rand
        pct = d / b_rand * 100
        print(f'  2. BERT: AAN hard vs AAN random:      {d:+.4f} ({pct:+.1f}%) '
              f'→ {"MEJORA" if d>0 else "DEGRADACIÓN"}')

    # 3. Efecto del encoder: RoBERTa vs BERT baseline
    if b_base and r_base:
        d = r_base - b_base
        pct = d / b_base * 100
        print(f'  3. RoBERTa vs BERT (baseline):        {d:+.4f} ({pct:+.1f}%) '
              f'→ {"MEJORA" if d>0 else "DEGRADACIÓN"}')

    # 4. Efecto de AAN random sobre RoBERTa baseline
    if r_base and r_rand:
        d = r_rand - r_base
        pct = d / r_base * 100
        print(f'  4. RoBERTa: AAN random vs Baseline:   {d:+.4f} ({pct:+.1f}%) '
              f'→ {"MEJORA" if d>0 else "DEGRADACIÓN"}')

    # 5. Efecto del hard mining sobre AAN random (RoBERTa)
    if r_rand and r_hard:
        d = r_hard - r_rand
        pct = d / r_rand * 100
        print(f'  5. RoBERTa: AAN hard vs AAN random:   {d:+.4f} ({pct:+.1f}%) '
              f'→ {"MEJORA" if d>0 else "DEGRADACIÓN"}')

    # 6. Mejor resultado general
    best_row = df[df['Dataset']==ds].loc[df[df['Dataset']==ds]['Media'].idxmax()]
    print(f'  6. Mejor resultado: {best_row["Encoder"]} {best_row["Método"]} = {best_row["Media"]:.4f}')

## 5. Exportar resultados

In [ ]:
df_export = df.sort_values(['Dataset', 'Encoder', 'Método'])
df_export.to_csv(RES_DIR / 'all_results.csv', index=False)
print('Exportado a all_results.csv')
print()
print(df_export.to_string(index=False))
print()
print('NB4 completo.')